# Create HFRI Awards (Hellenic Foundation for Research and Innovation, ELIDEK)

Creates HFRI awards from the foundation's own published **funding lists**
("Κατάλογος Χρηματοδότησης" — the proposals approved for funding in each
call, plus numbered amendments when reserves move up). HFRI publishes them as
PDF announcements on its public call pages:

- current site https://www.elidek.gr/details/?id=N ("Δράσεις", one page per call)
- legacy site https://old.elidek.gr/call/... (calls 2017-2023)

The tables are grant-level: proposal number, title, PI (ΕΥ / ΜΕ-ΕΥ), host
institution, [partner institutions], approved budget (EUR). Ranking tables
(number + grade only) are not used. The login-only grant portal
(portal.hfri.gr/Projects, named in the tracker) is **not** used.

**Coverage (local run 2026-09-30): 1,186 awards, EUR 200.5M, 99.9% title,
99.3% amount, 78% PI name, 100% host institution**, from 137 funding-list
PDFs across 15 calls: 1st/2nd/3rd Faculty-Members calls (281/154/86), Basic
Research Greece 2.0 / RRF (268), 5th PhD scholarships (185), 4th Post-doc
(115), OFYPEKA biodiversity studies + PhD (60), Science & Society actions,
equipment, NEH-HFRI. The 1st Faculty-Members lists print no PI names
(title/host/budget only), hence the PI gap. **Not covered:** the 2016-2019
PhD (1st-4th) and Post-doc (1st-3rd) calls (short, per-call proposal numbers
with no citable form; ~900 rows dropped) and calls whose lists are not
published as text PDFs. No call in the lists is flagged "(ΓΓΕΚ)", so no rows
route to GSRT from this source.

**Prerequisites:**
- Run `scripts/local/hfri_to_s3.py` first. It uploads
  `s3://openalex-ingest/awards/hfri/hfri_projects.parquet` (§1.4 shrink guard).

**`funder_award_id` (§2.1.1):** the HFRI proposal number, which is what
grantees cite (citation shells for F4320327859 are dominated by the bare
number, e.g. `16718`; the 1st and 2nd Faculty-Members calls are cited as
`HFRI-FM17-<n>` / `HFRI-FM20-<n>`, their official project codes, and ship in
that form). The 2016-2019 post-doc / PhD / Science-and-Society calls used
short per-call numbers that repeat across calls and have no citable
call-coded form, so those rows are **dropped** (logged by the script) rather
than given an invented key.

**Funder routing (§2.3.2):** HFRI (F4320327859) by default. A call HFRI ran
with General Secretariat for Research and Innovation money is flagged
"(ΓΓΕΚ)" in its title (2nd PhD-scholarship call); its rows route to
**GSRT** (F4320321917), the OpenAlex funder for the GSRT/GSRI secretariat.
Column `funder_key` in the parquet (`hfri` / `gsrt`).

**Dates:** the lists carry no dates; `start_date`/`end_date` NULL.

**Priority:** `492` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.hfri_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/hfri/hfri_projects.parquet`;

In [ ]:
%sql
SELECT funder_key, `call`, COUNT(*) as n FROM openalex.awards.hfri_raw GROUP BY ALL ORDER BY n DESC;

In [ ]:
%sql
DESCRIBE openalex.awards.hfri_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.hfri_raw LIMIT 5;

## Step 1.6: Funder existence check
Both funders are Crossref-registered (F4320*): each MUST resolve to exactly 1
row in `openalex.funders.funders`. If not, STOP and flag.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.funders.funders
WHERE funder_id IN (4320327859, 4320321917);

## Step 2: Create HFRI Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.hfri_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id IN (4320327859, 4320321917)
),

parsed AS (
    SELECT
        g.*,
        -- §2.3.2: GSRI-funded calls ("(ΓΓΕΚ)" in the call title) -> GSRT
        CASE WHEN g.funder_key = 'gsrt' THEN 4320321917 ELSE 4320327859 END AS resolved_funder_id
    FROM openalex.awards.hfri_raw g
    WHERE g.funder_award_id IS NOT NULL AND TRIM(g.funder_award_id) != ''
)

SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000 as id,
    NULLIF(TRIM(g.title), '') as display_name,
    CAST(NULL AS STRING) as description,
    f.funder_id,
    TRIM(g.funder_award_id) as funder_award_id,
    TRY_CAST(g.amount AS DOUBLE) as amount,
    CASE WHEN TRY_CAST(g.amount AS DOUBLE) IS NOT NULL THEN 'EUR' END as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    CASE WHEN g.`call` RLIKE 'Υποτροφ' THEN 'fellowship'
         WHEN g.`call` RLIKE 'εξοπλισμού' THEN 'equipment'
         ELSE 'research' END as funding_type,
    NULLIF(TRIM(g.`call`), '') as funder_scheme,
    'hfri_funding_lists' as provenance,
    CAST(NULL AS DATE) as start_date,
    CAST(NULL AS DATE) as end_date,
    CAST(NULL AS INT) as start_year,
    CAST(NULL AS INT) as end_year,
    CASE WHEN NULLIF(TRIM(g.lead_family_name), '') IS NOT NULL THEN named_struct(
        'given_name', NULLIF(TRIM(g.lead_given_name), ''),
        'family_name', NULLIF(TRIM(g.lead_family_name), ''),
        'orcid', CAST(NULL AS STRING),
        'role_start', CAST(NULL AS DATE),
        'affiliation', named_struct(
            'name', NULLIF(TRIM(g.institution), ''),
            'country', 'GR',
            'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
        )
    ) END as lead_investigator,
    CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>) as co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>>) as investigators,
    COALESCE(g.call_source, g.doc_url) as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM parsed g
JOIN src_funder f ON f.funder_id = g.resolved_funder_id;

In [ ]:
%sql
-- funder_award_id shape check (§2.1.1): bare proposal number, or HFRI-FM17-<n>
-- for the 1st faculty-members call. Zero rows may carry any other form.
SELECT
    COUNT(*) as total,
    SUM(CASE WHEN funder_award_id RLIKE '^[0-9]{2,6}$' THEN 1 ELSE 0 END) as bare_number,
    SUM(CASE WHEN funder_award_id RLIKE '^HFRI-FM(17|20)-[0-9]+$' THEN 1 ELSE 0 END) as fm_calls,
    COLLECT_SET(CASE WHEN NOT funder_award_id RLIKE '^([0-9]{2,6}|HFRI-FM(17|20)-[0-9]+)$' THEN funder_award_id END) as odd_ids
FROM openalex.awards.hfri_awards;

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'hfri_funding_lists' AND priority = 492;

-- Insert into openalex_awards_raw with priority.
-- Priority 492: direct-from-funder ingest of HFRI's published funding lists.
-- Higher wins under the 2026-06-20 DESC dedup (oxjob #500).
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    492 as priority
FROM openalex.awards.hfri_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_hfri_awards, COUNT(DISTINCT id) as distinct_ids
FROM openalex.awards.hfri_awards;

In [ ]:
%sql
-- §6.5 funder split: HFRI plus the GSRI-funded call routed to GSRT.
SELECT funder.display_name, funder_id, COUNT(*) as n
FROM openalex.awards.hfri_awards
GROUP BY ALL ORDER BY n DESC;

In [ ]:
%sql
SELECT funder_scheme, funding_type, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_eur
FROM openalex.awards.hfri_awards
GROUP BY ALL ORDER BY cnt DESC;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, amount, currency,
       lead_investigator.given_name, lead_investigator.family_name,
       lead_investigator.affiliation.name
FROM openalex.awards.hfri_awards LIMIT 20;

In [ ]:
%sql
-- Section 6.4a frequency check: PI top-20 must be a real long-tail.
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.hfri_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT display_name, COUNT(*) AS n
FROM openalex.awards.hfri_awards
GROUP BY 1 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- Section 6.3 / 6.7 coverage.
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(amount) as has_amount,
    COUNT(DISTINCT currency) as distinct_currencies,
    ROUND(MIN(amount), 0) as min_amt,
    ROUND(MAX(amount), 0) as max_amt,
    ROUND(AVG(amount), 0) as avg_amt,
    COUNT(lead_investigator.family_name) as has_pi_name,
    COUNT(lead_investigator.affiliation.name) as has_institution
FROM openalex.awards.hfri_awards;

In [ ]:
%sql
-- Overlap with existing citation shells (priority 0) that this ingest upgrades.
SELECT c.funder_id, COUNT(DISTINCT c.id) as shell_ids, COUNT(DISTINCT t.id) as upgraded_by_hfri
FROM openalex.awards.openalex_awards_raw c
LEFT JOIN openalex.awards.hfri_awards t ON t.id = c.id
WHERE c.funder_id IN (4320327859, 4320321917) AND c.priority = 0
GROUP BY c.funder_id;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, funder_id, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'hfri_funding_lists'
GROUP BY ALL;